In [ ]:
import torch
import numpy as np
from torch.nn import *
from PIL.Image import open, fromarray, LANCZOS
from IPython.display import clear_output
from tqdm.notebook import tqdm


class F(Module):
    def __init__(s, f):
        super().__init__()
        s.f = f

    def forward(s, x):
        return s.f(x)


torch.set_default_tensor_type(torch.cuda.FloatTensor)
x = torch.stack(
    torch.broadcast_tensors(
        torch.linspace(-1, 1, 180)[:, None], torch.linspace(-1, 1, 320)
    ),
    2,
)
y = (
    torch.tensor(np.array(open("assets/a.png").resize((320, 180), LANCZOS)))
    .to("cuda", torch.float)
    .div_(255.999)
)
m = Sequential(Linear(2, 256), F(torch.sinh_), Linear(256, 3))
c = MSELoss()
o = torch.optim.Adam(m.parameters(), 1e-2)
while 1:
    for _ in range(10000):
        __ = m(x).sigmoid_()
        _ = c(__, y).mul_(65536)
        _.backward()
        o.step()
        o.zero_grad()
    clear_output(1)
    display(fromarray(__.detach().mul_(255.999).to("cpu", torch.uint8).numpy()))
    print(_)